In [ ]:
# ==============================================================================
# CELDA 1: MONTAJE DE GOOGLE DRIVE Y VERIFICACIÓN DE RUTAS
# ==============================================================================
import os
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# Montar Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print('✓ Google Drive ya está montado.')

DATASET_DIR = '/content/drive/MyDrive/GastroAI/dataset'
print(f"Ruta principal de escaneo: {DATASET_DIR}")


In [ ]:
# ==============================================================================
# CELDA 2: AUDITORÍA DE TAXONOMÍA ROBUSTA CON RECONOCIMIENTO MULTIFORMATO DE EGID
# ==============================================================================
def read_egid_metadata(meta_file):
    """Lee metadata.xlsx probando CSV, openpyxl, xlrd y pandas estándar."""
    if not os.path.exists(meta_file):
        return None
    
    # Intento 1: CSV (Muy común en archivos renombrados de Figshare)
    try:
        df_csv = pd.read_csv(meta_file)
        if len(df_csv.columns) >= 2 and len(df_csv) > 0:
            print(f"  ✓ metadata.xlsx leído exitosamente como CSV ({len(df_csv)} filas).")
            return df_csv
    except Exception:
        pass
        
    # Intento 2: openpyxl (.xlsx moderno)
    try:
        df_open = pd.read_excel(meta_file, engine='openpyxl')
        print(f"  ✓ metadata.xlsx leído exitosamente con openpyxl ({len(df_open)} filas).")
        return df_open
    except Exception:
        pass
        
    # Intento 3: xlrd (.xls antiguo)
    try:
        df_xlrd = pd.read_excel(meta_file, engine='xlrd')
        print(f"  ✓ metadata.xlsx leído exitosamente con xlrd ({len(df_xlrd)} filas).")
        return df_xlrd
    except Exception:
        pass
        
    # Intento 4: read_excel por defecto
    try:
        df_std = pd.read_excel(meta_file)
        print(f"  ✓ metadata.xlsx leído exitosamente ({len(df_std)} filas).")
        return df_std
    except Exception as e:
        print(f"  ⚠️ No se pudo procesar metadata.xlsx con ningún motor: {e}")
        return None

def verify_taxonomy_v3(dataset_dir):
    print("[SYNC] Sincronizando árbol de directorios de Google Drive...")
    # Forzar al driver FUSE de Colab a sincronizar subdirectorios profundos
    _ = glob.glob(os.path.join(dataset_dir, '**', '*'), recursive=True)
    
    available_dirs = {}
    if os.path.exists(dataset_dir):
        for d in os.listdir(dataset_dir):
            full_d = os.path.join(dataset_dir, d)
            if os.path.isdir(full_d):
                available_dirs[d.lower()] = full_d
                
    print("Carpetas de datasets detectadas en /dataset/:", list(available_dirs.keys()))
    records = []
    IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp')
    
    # -------------------------------------------------------------------------
    # 1. PROCESAR EGID
    # -------------------------------------------------------------------------
    egid_keys = [k for k in available_dirs.keys() if 'egid' in k]
    if egid_keys:
        egid_dir = available_dirs[egid_keys[0]]
        print(f"\n🔍 [EGID DETECTADO] Escaneando desde: {egid_dir}")
        
        meta_file = os.path.join(egid_dir, 'metadata.xlsx')
        df_meta = read_egid_metadata(meta_file)
        egid_meta_map = {}
        if df_meta is not None:
            df_meta.columns = [str(c).strip().lower() for c in df_meta.columns]
            for _, row in df_meta.iterrows():
                img_k = str(row.get('image_id', row.get('image', row.get('filename', '')))).strip()
                pat_k = str(row.get('patient_id', row.get('patient', 'PAT_EGID_UNK'))).strip()
                if img_k:
                    egid_meta_map[img_k] = pat_k
        
        img_count_egid = 0
        for root, _, files in os.walk(egid_dir):
            root_lower = root.lower()
            for f in files:
                if f.lower().endswith(IMAGE_EXTS):
                    img_count_egid += 1
                    full_path = os.path.join(root, f)
                    parts = full_path.split(os.sep)
                    
                    # Extraer ID de paciente
                    patient_id = egid_meta_map.get(f)
                    if not patient_id:
                        num_parts = [p for p in parts if p.isdigit()]
                        patient_id = f"PAT_EGID_{num_parts[-1]}" if num_parts else f"PAT_EGID_{img_count_egid % 200:03d}"
                    
                    hpyl = 1 if ('positive' in root_lower or 'h_pylori' in root_lower or '1.h_pylori' in root_lower) else 0
                    atrophy = 1 if ('atrophy' in root_lower or '2.presence_of_atrophy' in root_lower or '3.distribution' in root_lower) else 0
                    gastritis = 1 if ('gastritis' in root_lower or hpyl == 1) else 0
                    normal = 1 if (hpyl == 0 and atrophy == 0 and gastritis == 0) else 0
                    
                    records.append({
                        'dataset': 'EGID',
                        'image_id': f,
                        'patient_id': patient_id,
                        'normal_mucosa': normal,
                        'erythematous_gastritis': gastritis,
                        'gastric_atrophy': atrophy,
                        'h_pylori_signs': hpyl
                    })
        print(f"  ✓ EGID procesado exitosamente: {img_count_egid} imágenes encontradas.")
    else:
        print("⚠️ No se encontró ninguna carpeta coincidente con 'egid' en /dataset/.")
        
    # -------------------------------------------------------------------------
    # 2. PROCESAR GASTROHUN
    # -------------------------------------------------------------------------
    gastro_keys = [k for k in available_dirs.keys() if 'gastro' in k and 'egid' not in k]
    if gastro_keys:
        gastro_dir = available_dirs[gastro_keys[0]]
        print(f"\n🔍 [GASTROHUN DETECTADO] Escaneando desde: {gastro_dir}")
        img_count_gh = 0
        for root, _, files in os.walk(gastro_dir):
            for f in files:
                if f.lower().endswith(IMAGE_EXTS):
                    img_count_gh += 1
                    full_path = os.path.join(root, f)
                    rel_path = os.path.relpath(full_path, gastro_dir)
                    folder_part = rel_path.split(os.sep)[0]
                    patient_id = f"PAT_GH_{folder_part}"
                    
                    records.append({
                        'dataset': 'GastroHUN',
                        'image_id': f,
                        'patient_id': patient_id,
                        'normal_mucosa': 1,
                        'erythematous_gastritis': 0,
                        'gastric_atrophy': 0,
                        'h_pylori_signs': 0
                    })
        print(f"  ✓ GastroHUN procesado exitosamente: {img_count_gh} imágenes encontradas.")
        
    # -------------------------------------------------------------------------
    # 3. PROCESAR HYPERKVASIR (UPPER GI / ESTÓMAGO)
    # -------------------------------------------------------------------------
    hyper_keys = [k for k in available_dirs.keys() if 'hyper' in k or 'kvasir' in k]
    if hyper_keys:
        hyper_dir = available_dirs[hyper_keys[0]]
        print(f"\n🔍 [HYPERKVASIR DETECTADO] Escaneando (solo Upper GI) desde: {hyper_dir}")
        img_count_hk = 0
        for root, _, files in os.walk(hyper_dir):
            root_lower = root.lower()
            if any(k in root_lower for k in ['lower-gi-tract', 'colon', 'polyps', 'cecum']):
                continue
                
            for f in files:
                if f.lower().endswith(IMAGE_EXTS):
                    img_count_hk += 1
                    img_stem = os.path.splitext(f)[0]
                    patient_id = f"PAT_HK_{img_stem[:8]}"
                    
                    is_normal = any(k in root_lower for k in ['pylorus', 'retroflex', 'z-line'])
                    is_gastritis = 'esophagitis' in root_lower
                    is_atrophy = 'barrett' in root_lower
                    
                    normal = 1 if (is_normal and not is_gastritis and not is_atrophy) else 0
                    gastritis = 1 if is_gastritis else 0
                    atrophy = 1 if is_atrophy else 0
                    
                    records.append({
                        'dataset': 'HyperKvasir',
                        'image_id': f,
                        'patient_id': patient_id,
                        'normal_mucosa': normal,
                        'erythematous_gastritis': gastritis,
                        'gastric_atrophy': atrophy,
                        'h_pylori_signs': 0
                    })
        print(f"  ✓ HyperKvasir procesado exitosamente: {img_count_hk} imágenes gástricas encontradas.")
        
    df_audit = pd.DataFrame(records)
    return df_audit

df_audit_v3 = verify_taxonomy_v3(DATASET_DIR)


In [ ]:
# ==============================================================================
# CELDA 3: TABLA DE RESUMEN Y GRÁFICA DE DISTRIBUCIÓN DE TAXONOMÍA
# ==============================================================================
target_cols = ['normal_mucosa', 'erythematous_gastritis', 'gastric_atrophy', 'h_pylori_signs']

if len(df_audit_v3) > 0:
    summary_df = df_audit_v3.groupby('dataset')[target_cols].sum()
    summary_df['Total_Imágenes'] = df_audit_v3.groupby('dataset')['image_id'].count()
    summary_df['Pacientes_Únicos'] = df_audit_v3.groupby('dataset')['patient_id'].nunique()

    print("="*75)
    print(" 📊 AUDITORÍA DE DISTRIBUCIÓN DE CLASES Y PACIENTES (v3 EGID Incluido)")
    print("="*75)
    display(summary_df)

    print("\n--- PORCENTAJE DE REPRESENTACIÓN POR CLASE (%) ---")
    pct_df = (summary_df[target_cols].div(summary_df['Total_Imágenes'], axis=0) * 100).round(2)
    display(pct_df)

    # Gráfica comparativa visual
    plt.figure(figsize=(12, 6))
    df_melted = df_audit_v3.melt(id_vars=['dataset'], value_vars=target_cols, var_name='Clase_Patológica', value_name='Presente')
    df_melted = df_melted[df_melted['Presente'] == 1]

    sns.countplot(data=df_melted, x='Clase_Patológica', hue='dataset', palette='Set2')
    plt.title('Distribución de Clases Patológicas Unificadas por Dataset (GastroAI v3)', fontsize=14, fontweight='bold')
    plt.xlabel('Clase de la Cascada de Correa', fontsize=12)
    plt.ylabel('Cantidad de Muestras Positivas', fontsize=12)
    plt.xticks(rotation=15)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No se encontraron imágenes para auditar. Verifique la ruta DATASET_DIR.")
